# Bonus Calculation Automation

This notebook automates the monthly bonus calculation process for an equipment rental business.

## Load Operational Data

Data is retrieved from Google Sheets in the production environment.

In [34]:
!pip install gspread gspread_dataframe -q

from google.colab import auth
auth.authenticate_user()

import pandas as pd
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

creds, _ = default()
gc = gspread.authorize(creds)

In [35]:
# configuration

URL_PLANILHA = None # Connection details omitted for confidentiality

ABA_MOVIMENTACOES = "Movimentaçoes"
ABA_CLIENTES = "Clientes"
ABA_OBRAS = "Obras"
ABA_MANUTENÇOES = "Troca de oleo"
ABA_MAQUINAS = "Maquinas"
ABA_STATUS = "Atualizaçao horimetro"

INICIO_MES = pd.Timestamp("2026-05-01")
FIM_MES = pd.Timestamp("2026-05-31")

In [36]:
# read machines table

spreadsheet = gc.open_by_url(URL_PLANILHA)
maq = spreadsheet.worksheet(ABA_MAQUINAS)

maquinas = pd.DataFrame(maq.get_all_records())

In [37]:
# keep only relevant columns
maquinas = maquinas[
    [
        "codigo_maquina",
        "tipo_maquina"
    ]
]

In [38]:
# read movements table

ws = spreadsheet.worksheet(ABA_MOVIMENTACOES)

df = pd.DataFrame(ws.get_all_records())


In [39]:
# fix date data type

df["Data"] = pd.to_datetime(df["Data"], dayfirst=True, errors="coerce")
df["Data_mediçao"] = pd.to_datetime(df["Data_mediçao"], dayfirst=True, errors="coerce")

if "Criado_em" in df.columns:
    df["Criado_em"] = pd.to_datetime(df["Criado_em"], dayfirst=True, errors="coerce")
else:
    df["Criado_em"] = df["Data"]

# remove not active lines
if "Status_registro" in df.columns:
    df = df[df["Status_registro"].fillna("Ativo") != "Cancelado"]

In [40]:
# get correct date
df = df.copy()

df["data_evento"] = df["Data"]

condicao_data_nao_coincide = (
    df["Data_coincide"]
    .astype(str)
    .str.strip()
    .str.upper()
    .isin(["FALSE", "FALSO", "0", "NO", "NÃO", "NAO"])
)

df.loc[condicao_data_nao_coincide, "data_evento"] = df["Data_mediçao"]

In [41]:
# reestructure table so machines and dates will always appear in the same column regardless of movement type
eventos = []

for _, row in df.iterrows():

    tipo = row["Tipo_Movimento"]

    if tipo in ["Saída de máquina", "Troca de máquina"]:
        eventos.append({
            "maquina": row["Maquina_Saida"],
            "tipo_evento": "saida",
            "data": row["data_evento"],
            "criado_em": row["Criado_em"],
            "cliente": row["Cliente"],
            "obra": row["Obra"],
            "cobrar_frete_saida": row["Frete_Cobrar"],
            "movimento_origem_saida": tipo,
            "horimetro_saida": row["Horimetro_Saida"]
        })

    if tipo in ["Retorno de máquina", "Troca de máquina"]:
        eventos.append({
            "maquina": row["Maquina_Entrada"],
            "tipo_evento": "entrada",
            "data": row["data_evento"],
            "criado_em": row["Criado_em"],
            "cliente": row["Cliente"],
            "obra": row["Obra"],
            "cobrar_frete_retorno": row["Frete_Cobrar"],
            "movimento_origem_retorno": tipo,
            "horimetro_retorno": row["Horimetro_Entrada"]
        })

eventos = pd.DataFrame(eventos)

eventos = eventos.dropna(subset=["maquina", "data"])
eventos = eventos[eventos["maquina"].astype(str).str.strip() != ""]

eventos = eventos.sort_values(["maquina", "data", "criado_em"])

In [42]:
# reorganize table so we have each rental operation per machine, both current and past rentals
locacoes = []

for maquina, grupo in eventos.groupby("maquina"):

    locacao_aberta = None

    for _, ev in grupo.iterrows():

        if ev["tipo_evento"] == "saida":

            locacao_aberta = {
                "maquina": maquina,
                "cliente": ev["cliente"],
                "obra": ev["obra"],
                "data_saida": ev["data"],
                "frete_saida": ev.get("cobrar_frete_saida", None),
                "movimento_saida": ev.get("movimento_origem_saida", None),
                "horimetro_saida": ev["horimetro_saida"]
            }

        elif ev["tipo_evento"] == "entrada" and locacao_aberta is not None:

            locacao_aberta["data_retorno"] = ev["data"]
            locacao_aberta["frete_retorno"] = ev.get("cobrar_frete_retorno", None)
            locacao_aberta["movimento_retorno"] = ev.get("movimento_origem_retorno", None)
            locacao_aberta["horimetro_retorno"] = ev.get("horimetro_retorno", None)

            locacoes.append(locacao_aberta)
            locacao_aberta = None

    if locacao_aberta is not None:
        locacao_aberta["data_retorno"] = pd.NaT
        locacao_aberta["frete_retorno"] = None
        locacao_aberta["movimento_retorno"] = None
        locacoes.append(locacao_aberta)

locacoes = pd.DataFrame(locacoes)

In [43]:
# filter for location period on the month we are analysing

resultado = locacoes[
    (locacoes["data_saida"] <= FIM_MES) &
    (
        locacoes["data_retorno"].isna() |
        (locacoes["data_retorno"] >= INICIO_MES)
    )
].copy()



In [44]:
# calculate how long they were rented on the analysed month

resultado["inicio_cobranca_mes"] = resultado["data_saida"].apply(
    lambda x: max(x, INICIO_MES)
)

resultado["fim_cobranca_mes"] = resultado["data_retorno"].apply(
    lambda x: min(x, FIM_MES) if pd.notna(x) else FIM_MES
)

resultado["dias_no_mes"] = (
    resultado["fim_cobranca_mes"] -
    resultado["inicio_cobranca_mes"]
).dt.days + 1

resultado = resultado[
    [
        "maquina",
        "cliente",
        "obra",
        "data_saida",
        "data_retorno",
        "inicio_cobranca_mes",
        "fim_cobranca_mes",
        "dias_no_mes",
        "frete_saida",
        "frete_retorno",
        "horimetro_saida",
        "horimetro_retorno"
    ]
].sort_values(["cliente", "obra", "maquina"])



## Join with clients table:

In [45]:
# read clients table
clients = spreadsheet.worksheet(ABA_CLIENTES)

clientes = pd.DataFrame(clients.get_all_records())

In [46]:
# Join with clients table

resultado_final = pd.merge(
    resultado,
    clientes[["Cliente_ID", "Cliente_Nome"]],
    left_on="cliente",
    right_on="Cliente_ID",
    how="left"
)

resultado_final = resultado_final.drop(
    columns=["Cliente_ID", "cliente"]
)


In [47]:
def resumir_clientes(grupo):
    grupo = grupo.sort_values("data_saida")

    partes = [
        f"{row['Cliente_Nome']} ({row['dias_no_mes']} dias)"
        for _, row in grupo.iterrows()
    ]

    return " / ".join(partes)


dias_periodo = (FIM_MES - INICIO_MES).days + 1

resumo_maquinas = (
    resultado_final
    .groupby("maquina", as_index=False)
    .agg(
        dias_no_mes=("dias_no_mes", "sum"),
        Cliente_Nome=("Cliente_Nome", lambda x: resumir_clientes(resultado_final.loc[x.index]))
    )
)

resumo_maquinas["dias_no_mes"] = resumo_maquinas["dias_no_mes"].clip(upper=dias_periodo)


##Join with machines table

In [48]:
resumo_final = pd.merge(
    maquinas,
    resumo_maquinas,
    left_on="codigo_maquina",
    right_on="maquina",
    how="left"
)

In [49]:
resumo_final = resumo_final.drop(columns=["maquina"])

add the machine's status during the analysed period:

In [50]:
#read status table
atualizacoes = spreadsheet.worksheet(ABA_STATUS)

status = pd.DataFrame(atualizacoes.get_all_records())

In [51]:
# get only valid status updates
status = status[
    (status["Tipo_atualizaçao"] == "Atualizaçao de status de máquina") &
    (status["Validez"]!= "FALSE")
].copy()

In [52]:
status_hist = status.copy()

status_hist["Data"] = pd.to_datetime(status_hist["Data"], dayfirst=True, errors="coerce")

# select only status relevant to the analysed period
status_hist = status_hist[
    status_hist["Data"] <= FIM_MES
].copy()

def resumo_status_mes(grupo):
    grupo = grupo.sort_values("Data").copy()


    antes = grupo[grupo["Data"] <= INICIO_MES].tail(1)


    dentro = grupo[
        (grupo["Data"] > INICIO_MES) &
        (grupo["Data"] <= FIM_MES)
    ]

    eventos = pd.concat([antes, dentro]).sort_values("Data")

    if eventos.empty:
        return pd.Series({
            "status_no_mes": "",
            "dias_por_status": 0
        })

    periodos = []

    for i, row in eventos.reset_index(drop=True).iterrows():
        data_inicio = max(row["Data"], INICIO_MES)

        if i < len(eventos) - 1:
            data_fim = eventos.reset_index(drop=True).loc[i + 1, "Data"] - pd.Timedelta(days=1)
        else:
            data_fim = FIM_MES

        data_fim = min(data_fim, FIM_MES)

        dias = (data_fim - data_inicio).days + 1

        if dias > 0:
            periodos.append({
                "Status": row["Status"],
                "dias": dias,
                "data_inicio": data_inicio
            })

    resumo = (
        pd.DataFrame(periodos)
        .groupby("Status", as_index=False)["dias"]
        .sum()
    )

    texto = " / ".join(
        f"{row['Status']} ({int(row['dias'])} dias)"
        for _, row in resumo.iterrows()
    )

    return pd.Series({
        "status_no_mes": texto,
        "dias_por_status": resumo["dias"].sum()
    })


resumo_status = (
    status_hist
    .groupby("Máquina")
    .apply(resumo_status_mes)
    .reset_index()
    .rename(columns={"Máquina": "codigo_maquina"})
)


/tmp/ipykernel_3407/985085326.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resumo_status_mes)


In [53]:
#merge with resultado_final
resumo_final = pd.merge(
    resumo_final,
    resumo_status,
    on="codigo_maquina",
    how="left"
)


In [54]:
resumo_final = resumo_final.drop(columns=["dias_por_status"])

filter this df to only have generators and electrical towers since they are the only ones relevant for bonus calculations.

In [55]:
resumo_geradores_torres = resumo_final[
    resumo_final["tipo_maquina"].isin(["Gerador", "Torre"])
].copy()

## Calculate bonus:

In [56]:
dias_mes = (FIM_MES - INICIO_MES).days + 1

In [57]:
valor_diaria = 35/dias_mes

In [58]:
resumo_geradores_torres["gratificacao"] = (resumo_geradores_torres["dias_no_mes"] * valor_diaria).round(2)

add a line on the end with total bonus value.

In [59]:
total_gratificacao = resumo_geradores_torres["gratificacao"].sum()

total_row = pd.DataFrame([{
    "codigo_maquina": "Total",
    "gratificacao": total_gratificacao
}])

resumo_geradores_torres = pd.concat([resumo_geradores_torres, total_row], ignore_index=True)

calculate the bonus per employee.

In [60]:
df_gratificacao = pd.DataFrame({
    "Nome": ["Abby", "Beth", "Carlo", "Dora", "Emily"],
    "Percentual": [0.50, 0.05, 0.05, 0.175, 0.175]
})

df_gratificacao["Valor"] = (
    df_gratificacao["Percentual"] * total_gratificacao
).round(2)

df_gratificacao["Percentual_%"] = (
    df_gratificacao["Percentual"] * 100
).round(1)

display(df_gratificacao)

,Nome,Percentual,Valor,Percentual_%
0,Abby,0.500,2208.39,50.0
1,Beth,0.050,220.84,5.0
2,Carlo,0.050,220.84,5.0
3,Dora,0.175,772.93,17.5
4,Emily,0.175,772.93,17.5


now a df with only equipment that is neither generators nor towers.

In [61]:
resumo_outros_equipamentos = resumo_final[
    ~resumo_final["tipo_maquina"].isin(["Gerador", "Torre"])
].copy()

In [62]:
resumo_outros_equipamentos = resumo_outros_equipamentos.drop(columns=["status_no_mes"])

another df to calculate the number of generators and towers per status.

In [63]:
# correct dates
status = status.copy()
maquinas = maquinas.copy()

status["Data"] = pd.to_datetime(status["Data"], errors="coerce")
status["Criado_em"] = pd.to_datetime(status["Criado_em"], errors="coerce")

# filter machine type
maquinas_base = maquinas[
    maquinas["tipo_maquina"].isin(["Gerador", "Torre"])
].copy()

# get relevant status
status_ate_fim = status[
    status["Data"] <= FIM_MES
].copy()

# take the last status of the analysed period
ultimo_status = (
    status_ate_fim
    .sort_values(["Máquina", "Data", "Criado_em"])
    .groupby("Máquina", as_index=False)
    .tail(1)
)

# merge machines and last status
maquinas_status_fim_mes = maquinas_base.merge(
    ultimo_status[["Máquina", "Status", "Data"]],
    left_on="codigo_maquina",
    right_on="Máquina",
    how="left"
)

# if a machine doesn't have a status
maquinas_status_fim_mes["Status"] = maquinas_status_fim_mes["Status"].fillna("Sem status")

# count machines per status
df_status_fim_mes = (
    maquinas_status_fim_mes
    .groupby("Status", as_index=False)
    .agg(quantidade_maquinas=("codigo_maquina", "count"))
    .sort_values("quantidade_maquinas", ascending=False)
)

linha_total = pd.DataFrame({
    "Status": ["TOTAL"],
    "quantidade_maquinas": [df_status_fim_mes["quantidade_maquinas"].sum()]
})

df_status_fim_mes = pd.concat(
    [df_status_fim_mes, linha_total],
    ignore_index=True
)


/tmp/ipykernel_3407/1147664690.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  status["Data"] = pd.to_datetime(status["Data"], errors="coerce")
/tmp/ipykernel_3407/1147664690.py:6: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  status["Criado_em"] = pd.to_datetime(status["Criado_em"], errors="coerce")


In [64]:
df_status_fim_mes = (
    maquinas_status_fim_mes
    .groupby(
        ["tipo_maquina", "Status"],
        as_index=False
    )
    .agg(
        quantidade_maquinas=("codigo_maquina", "count")
    )
    .sort_values(
        ["tipo_maquina", "quantidade_maquinas"],
        ascending=[True, False]
    )
)

In [65]:
totais = (
    df_status_fim_mes
    .groupby("tipo_maquina", as_index=False)
    .agg(
        quantidade_maquinas=("quantidade_maquinas", "sum")
    )
)

totais["Status"] = "TOTAL"

df_status_fim_mes = pd.concat(
    [df_status_fim_mes, totais],
    ignore_index=True
)

df_status_fim_mes = df_status_fim_mes.sort_values(
    ["tipo_maquina", "Status"]
)


## Generate report and export excel file

In [66]:
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# configurations

ARQ_RELATORIO = "Relatorio_Operacional_2026_05.xlsx"

cor_cabecalho = PatternFill("solid", fgColor="D9E2F3")
cor_titulo = PatternFill("solid", fgColor="1F4E78")
cor_sem_aluguel = PatternFill("solid", fgColor="E6B8B7")

fonte_titulo = Font(bold=True, color="FFFFFF", size=12)
fonte_cabecalho = Font(bold=True)
fonte_total = Font(bold=True)

borda_fina = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)

# dashboard tables

resumo_geradores_torres = resumo_geradores_torres.drop(
    columns=["index"],
    errors="ignore"
)

resumo_outros_equipamentos = resumo_outros_equipamentos.drop(
    columns=["index"],
    errors="ignore"
)

resumo_sem_total = resumo_geradores_torres[
    resumo_geradores_torres["codigo_maquina"].astype(str).str.upper() != "TOTAL"
].copy()

total_gratificacao = resumo_sem_total["gratificacao"].sum()
qtd_maquinas = resumo_sem_total["codigo_maquina"].nunique()

dias_mes = (FIM_MES - INICIO_MES).days + 1

total_dias_possiveis = qtd_maquinas * dias_mes

total_dias_locados = pd.to_numeric(
    resumo_sem_total["dias_no_mes"],
    errors="coerce"
).fillna(0).sum()

ocupabilidade = (
    total_dias_locados / total_dias_possiveis
    if total_dias_possiveis > 0
    else 0
)

dashboard_indicadores = pd.DataFrame({
    "Indicador": [
        "Máquinas analisadas",
        "Gratificação total",
        "Ocupabilidade"
    ],
    "Valor": [
        qtd_maquinas,
        total_gratificacao,
        ocupabilidade
    ]
})

df_gratificacao_dashboard = df_gratificacao.drop(
    columns=["Percentual_%"],
    errors="ignore"
).copy()

status_geradores = df_status_fim_mes[
    df_status_fim_mes["tipo_maquina"] == "Gerador"
].copy()

status_torres = df_status_fim_mes[
    df_status_fim_mes["tipo_maquina"] == "Torre"
].copy()

status_geradores = status_geradores.drop(
    columns=["tipo_maquina"],
    errors="ignore"
)

status_torres = status_torres.drop(
    columns=["tipo_maquina"],
    errors="ignore"
)

# paint lines with machines not rented during the analysed period

def pintar_linhas_sem_movimentacao(ws):
    col_dias = None

    for cell in ws[1]:
        if str(cell.value).strip() == "dias_no_mes":
            col_dias = cell.column
            break

    if col_dias is None:
        return

    for row in range(2, ws.max_row + 1):

        codigo_maquina = ws.cell(row=row, column=1).value
        valor_dias = ws.cell(row=row, column=col_dias).value

        linha_total = (
            codigo_maquina is not None
            and str(codigo_maquina).strip().upper() == "TOTAL"
        )

        sem_movimentacao = (
            valor_dias is None
            or str(valor_dias).strip() == ""
        )

        if sem_movimentacao and not linha_total:
            for col in range(1, ws.max_column + 1):
                ws.cell(row=row, column=col).fill = cor_sem_aluguel

        if linha_total:
            for col in range(1, ws.max_column + 1):
                ws.cell(row=row, column=col).font = fonte_total

# generate excel file

with pd.ExcelWriter(ARQ_RELATORIO, engine="openpyxl") as writer:

    resumo_geradores_torres.to_excel(
        writer,
        sheet_name="Resumo_Geradores_Torres",
        index=False
    )

    resumo_outros_equipamentos.to_excel(
        writer,
        sheet_name="Outros_Equipamentos",
        index=False
    )

    df_gratificacao_dashboard.to_excel(
        writer,
        sheet_name="Gratificacao",
        index=False
    )

    df_status_fim_mes.to_excel(
        writer,
        sheet_name="Status_Maquinas",
        index=False
    )

    wb = writer.book

    pintar_linhas_sem_movimentacao(
        writer.sheets["Resumo_Geradores_Torres"]
    )

    pintar_linhas_sem_movimentacao(
        writer.sheets["Outros_Equipamentos"]
    )


    # dashboard

    ws = wb.create_sheet("Dashboard", 0)

    ws["A1"] = "RELATÓRIO OPERACIONAL"
    ws["A1"].font = Font(bold=True, size=16)

    ws["A2"] = "Maio/2026"
    ws["A2"].font = Font(italic=True)

    linha = 4

    # KPIs

    ws[f"A{linha}"] = "Indicadores gerais"
    ws[f"A{linha}"].fill = cor_titulo
    ws[f"A{linha}"].font = fonte_titulo
    ws.merge_cells(start_row=linha, start_column=1, end_row=linha, end_column=2)

    linha += 1

    for col, nome_coluna in enumerate(dashboard_indicadores.columns, start=1):
        cell = ws.cell(row=linha, column=col)
        cell.value = nome_coluna
        cell.font = fonte_cabecalho
        cell.fill = cor_cabecalho
        cell.alignment = Alignment(horizontal="center")
        cell.border = borda_fina

    linha += 1

    for _, row in dashboard_indicadores.iterrows():
        ws.cell(row=linha, column=1).value = row["Indicador"]
        ws.cell(row=linha, column=2).value = row["Valor"]

        if row["Indicador"] == "Gratificação total":
            ws.cell(row=linha, column=2).number_format = 'R$ #,##0.00'

        if row["Indicador"] == "Ocupabilidade":
            ws.cell(row=linha, column=2).number_format = '0.0%'

        for col in range(1, 3):
            ws.cell(row=linha, column=col).border = borda_fina
            ws.cell(row=linha, column=col).alignment = Alignment(horizontal="center")

        linha += 1


    # Bonus distribution


    linha += 2

    ws[f"A{linha}"] = "Distribuição da gratificação"
    ws[f"A{linha}"].fill = cor_titulo
    ws[f"A{linha}"].font = fonte_titulo
    ws.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=len(df_gratificacao_dashboard.columns)
    )

    linha += 1

    for col, nome_coluna in enumerate(df_gratificacao_dashboard.columns, start=1):
        cell = ws.cell(row=linha, column=col)
        cell.value = nome_coluna
        cell.font = fonte_cabecalho
        cell.fill = cor_cabecalho
        cell.alignment = Alignment(horizontal="center")
        cell.border = borda_fina

    linha += 1

    for _, row in df_gratificacao_dashboard.iterrows():
        for col, nome_coluna in enumerate(df_gratificacao_dashboard.columns, start=1):
            cell = ws.cell(row=linha, column=col)
            cell.value = row[nome_coluna]
            cell.border = borda_fina
            cell.alignment = Alignment(horizontal="center")

            if nome_coluna == "Valor":
                cell.number_format = 'R$ #,##0.00'

            if nome_coluna == "Percentual":
                cell.number_format = '0.0%'

        linha += 1

    # =========================
    # get generators status
    # =========================

    linha += 2

    ws[f"A{linha}"] = "Status dos geradores no último dia do mês"
    ws[f"A{linha}"].fill = cor_titulo
    ws[f"A{linha}"].font = fonte_titulo
    ws.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=len(status_geradores.columns)
    )

    linha += 1

    for col, nome_coluna in enumerate(status_geradores.columns, start=1):
        cell = ws.cell(row=linha, column=col)
        cell.value = nome_coluna
        cell.font = fonte_cabecalho
        cell.fill = cor_cabecalho
        cell.alignment = Alignment(horizontal="center")
        cell.border = borda_fina

    linha += 1

    for _, row in status_geradores.iterrows():
        for col, nome_coluna in enumerate(status_geradores.columns, start=1):
            cell = ws.cell(row=linha, column=col)
            cell.value = row[nome_coluna]
            cell.border = borda_fina
            cell.alignment = Alignment(horizontal="center")

            if str(row.get("Status", "")).upper() == "TOTAL":
                cell.font = fonte_total

        linha += 1

    # =========================
    # get towers status
    # =========================

    linha += 2

    ws[f"A{linha}"] = "Status das torres no último dia do mês"
    ws[f"A{linha}"].fill = cor_titulo
    ws[f"A{linha}"].font = fonte_titulo
    ws.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=len(status_torres.columns)
    )

    linha += 1

    for col, nome_coluna in enumerate(status_torres.columns, start=1):
        cell = ws.cell(row=linha, column=col)
        cell.value = nome_coluna
        cell.font = fonte_cabecalho
        cell.fill = cor_cabecalho
        cell.alignment = Alignment(horizontal="center")
        cell.border = borda_fina

    linha += 1

    for _, row in status_torres.iterrows():
        for col, nome_coluna in enumerate(status_torres.columns, start=1):
            cell = ws.cell(row=linha, column=col)
            cell.value = row[nome_coluna]
            cell.border = borda_fina
            cell.alignment = Alignment(horizontal="center")

            if str(row.get("Status", "")).upper() == "TOTAL":
                cell.font = fonte_total

        linha += 1


    # format file

    for ws in wb.worksheets:

        ws.freeze_panes = "A2"

        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = Alignment(
                    horizontal="center",
                    vertical="center"
                )

        for col in ws.columns:
            max_length = 0
            col_letter = get_column_letter(col[0].column)

            for cell in col:
                if cell.value is not None:
                    texto = str(cell.value)
                    max_length = max(max_length, len(texto))

            ws.column_dimensions[col_letter].width = min(max_length + 6, 45)

print(f"Arquivo gerado: {ARQ_RELATORIO}")

Arquivo gerado: Relatorio_Operacional_2026_05.xlsx


## Results

The workflow automatically calculates machine rental days and generates the information required for the monthly bonus calculation.

This automation reduced a process that previously required manual review of equipment records and spreadsheet consolidation, reducing processing time from hours to seconds.